# TAMER OCR - High Performance Curriculum Training
Optimized for high VRAM/RAM utilization and reduced validation overhead.

In [13]:
import os
from getpass import getpass
from huggingface_hub import login

print("=========================================")
print("🤗 HUGGING FACE AUTHENTICATION")
print("=========================================")
print("Paste your Hugging Face Token (must have WRITE access).")
hf_token = getpass("HF Token: ").strip()
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=True)
print("✅ Authenticated successfully!")

🤗 HUGGING FACE AUTHENTICATION
Paste your Hugging Face Token (must have WRITE access).


HF Token:  ········


Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
✅ Authenticated successfully!


In [14]:
# IMPORTANT: Change 'YOUR_GITHUB_USERNAME' below to point to your repository
REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"

!git clone {REPO_URL}
%cd AI_PROJECT_TAMER_Complete/tamer_ocr

!pip install -r requirements.txt
!pip install datasets huggingface_hub

Cloning into 'AI_PROJECT_TAMER_Complete'...
remote: Enumerating objects: 284, done.
remote: Counting objects: 100% (284/284), done.
remote: Compressing objects: 100% (193/193), done.
remote: Total 284 (delta 142), reused 231 (delta 89), pack-reused 0 (from 0)
Receiving objects: 100% (284/284), 276.12 KiB | 4.93 MiB/s, done.
Resolving deltas: 100% (142/142), done.
/kaggle/working/AI_PROJECT_TAMER_Complete/tamer_ocr/AI_PROJECT_TAMER_Complete/tamer_ocr


In [15]:
import os
import re
from huggingface_hub import HfApi, hf_hub_download

# This pulls your highest epoch checkpoint so you can resume training
MODEL_REPO = "JJKK1212/tamer-math-ocr" 

os.makedirs('./checkpoints', exist_ok=True)
print("🔄 Checking for existing checkpoints on Hugging Face...")

try:
    api = HfApi(token=os.environ['HF_TOKEN'])
    # List all files in your HF repo
    files = api.list_repo_files(repo_id=MODEL_REPO, repo_type="model")
    
    # Find all files that match "checkpoint_epoch_X.pt"
    ckpt_files = [f for f in files if re.match(r"checkpoint_epoch_\d+\.pt", f)]
    
    if ckpt_files:
        # Sort them by the number in the filename and pick the highest one
        latest_remote_file = max(ckpt_files, key=lambda x: int(re.findall(r"\d+", x)[0]))
        print(f"📥 Found latest checkpoint: {latest_remote_file}. Downloading...")
        
        downloaded_path = hf_hub_download(
            repo_id=MODEL_REPO, 
            filename=latest_remote_file, 
            local_dir="./checkpoints", 
            repo_type="model",
            token=os.environ['HF_TOKEN']
        )
        
        # Rename it locally to latest.pt so train.py finds it
        local_latest_path = "./checkpoints/latest.pt"
        if os.path.exists(local_latest_path):
            os.remove(local_latest_path)
        os.rename(downloaded_path, local_latest_path)
        
        print("✅ Successfully downloaded and prepared! Training will resume.")
    else:
        print("ℹ️ No checkpoint_epoch_X.pt files found on Hub. Training from scratch (Epoch 0).")
        
except Exception as e:
    print(f"❌ Error checking Hub: {e}")

🔄 Checking for existing checkpoints on Hugging Face...
ℹ️ No checkpoint_epoch_X.pt files found on Hub. Training from scratch (Epoch 0).


### Stage 1: Printed Formulas (`im2latex-100k`)
Train until Epoch 15. Validation and saving happen only every 5 epochs.

In [16]:
!python train.py --datasets im2latex-100k --batch-size 64 --num-epochs 15 --save-every 5 --eval-every 5 --num-workers 4 --download

2026-04-09 01:24:05 - TAMER - INFO - Initialized TAMER Pipeline on device: cuda
2026-04-09 01:24:05 - TAMER - INFO - ============================================================
2026-04-09 01:24:05 - TAMER - INFO - PRE-TRAINING DATASET VALIDATION
2026-04-09 01:24:05 - TAMER - INFO - ============================================================
2026-04-09 01:24:05 - TAMER - INFO - Checking dataset: im2latex-100k
2026-04-09 01:24:05 - TAMER.Validator - INFO - ============================================================
2026-04-09 01:24:05 - TAMER.Validator - INFO - TRUSTED SOURCE FOUND: JJKK1212/Verified-Datasets-im2latex-100k
2026-04-09 01:24:05 - TAMER.Validator - INFO - Bypassing raw download/parsing and pulling directly from HF Hub.
2026-04-09 01:24:05 - TAMER.Validator - INFO - ============================================================
2026-04-09 01:24:07 - TAMER.Validator - INFO - Pulling verified dataset from Hugging Face: JJKK1212/Verified-Datasets-im2latex-100k...
README.md: 10

### Stage 2: Clean Handwritten (`mathwriting`)
Resume from Epoch 15 and train until Epoch 30.

In [ ]:
!python train.py --datasets mathwriting --batch-size 64 --num-epochs 30 --save-every 5 --eval-every 5 --num-workers 4 --download

2026-04-09 04:12:00 - TAMER - INFO - Initialized TAMER Pipeline on device: cuda
2026-04-09 04:12:00 - TAMER - INFO - ============================================================
2026-04-09 04:12:00 - TAMER - INFO - PRE-TRAINING DATASET VALIDATION
2026-04-09 04:12:00 - TAMER - INFO - ============================================================
2026-04-09 04:12:00 - TAMER - INFO - Checking dataset: mathwriting
2026-04-09 04:12:00 - TAMER.Validator - INFO - ============================================================
2026-04-09 04:12:00 - TAMER.Validator - INFO - TRUSTED SOURCE FOUND: JJKK1212/Verified-Datasets-mathwriting
2026-04-09 04:12:00 - TAMER.Validator - INFO - Bypassing raw download/parsing and pulling directly from HF Hub.
2026-04-09 04:12:00 - TAMER.Validator - INFO - ============================================================
2026-04-09 04:12:01 - TAMER.Validator - INFO - Pulling verified dataset from Hugging Face: JJKK1212/Verified-Datasets-mathwriting...
README.md: 100%|███

### Stage 3: Messy Handwritten (`crohme` & `hme100k`)
Resume from Epoch 30 and train until Epoch 60. This guarantees high real-world accuracy.

In [ ]:
!python train.py --datasets crohme hme100k --batch-size 64 --num-epochs 60 --save-every 5 --eval-every 5 --num-workers 4 --download